In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import Dense, Embedding, GlobalAveragePooling1D, Input
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.utils import to_categorical

# --- 1. Prepare the Dataset ---
corpus = [
    "the quick brown fox jumps over the lazy dog",
    "the quick brown dog jumps over the lazy fox",
    "a man is walking on the street",
    "a dog is barking at the cat",
    "the street is busy with people",
]

# --- 2. Tokenize and Create Word-to-Index Mappings ---
tokenizer = Tokenizer()
tokenizer.fit_on_texts(corpus)
word_index = tokenizer.word_index
vocab_size = len(word_index) + 1
print(f"Vocabulary Size: {vocab_size}")

# --- 3. Generate CBOW Training Data ---
sequences = tokenizer.texts_to_sequences(corpus)
window_size = 2
data = []

for sequence in sequences:
    for i, word in enumerate(sequence):
        # Create a list for context words (left and right within window)
        context = []
        for j in range(i - window_size, i + window_size + 1):
            if j != i and 0 <= j < len(sequence):
                context.append(sequence[j])
        if context:
            data.append((context, word))

# Separate context arrays (X) and target word arrays (y)
X_context = [item[0] for item in data]
y_target = np.array([item[1] for item in data])

# Pad the context sequences to ensure a uniform shape matrix
max_len = max(len(x) for x in X_context)
X_context_padded = pad_sequences(X_context, maxlen=max_len, padding="post")
y_target_one_hot = to_categorical(y_target, num_classes=vocab_size)

# --- 4. Build the CBOW Model (Functional API) ---
embedding_dim = 100

# Input placeholder layer
input_context = Input(shape=(max_len,), name="Context_Input")

# Embedding layer to convert word indices to dense continuous vectors
embedding_layer = Embedding(
    input_dim=vocab_size,
    output_dim=embedding_dim,
    name="Word_Embeddings",
)(input_context)

# FIXED: Replaced legacy Lambda + K.mean with GlobalAveragePooling1D
# This inherently calculates the mean across the sequence length (axis=1) safely
avg_context = GlobalAveragePooling1D(name="Average_Context")(embedding_layer)

# Output classification layer using Softmax activation
output_layer = Dense(
    vocab_size, activation="softmax", name="Target_Prediction"
)(avg_context)

# Create and define structural architecture bounds
cbow_model = Model(inputs=input_context, outputs=output_layer)

# --- 5. Compile and Train the Model ---
cbow_model.compile(
    optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"]
)
cbow_model.summary()

print("\nTraining the CBOW model...")
cbow_model.fit(X_context_padded, y_target_one_hot, epochs=500, verbose=0)
print("Training complete.")

# --- 6. Extract and Visualise Word Embeddings ---
# Safely fetch the embedding matrix weights by name layer reference
embedding_matrix = cbow_model.get_layer("Word_Embeddings").get_weights()[0]

print("\nLearned Word Embeddings (First 5 Dimensions shown):")
for word, idx in list(word_index.items())[:10]:  # Show first 10 words
    if idx < vocab_size:
        print(f"Word: '{word:<10}' Vector: {embedding_matrix[idx][:5]}")


Vocabulary Size: 21


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ Context_Input (InputLayer)      │ (None, 4)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Word_Embeddings (Embedding)     │ (None, 4, 100)         │         2,100 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Average_Context                 │ (None, 100)            │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Target_Prediction (Dense)       │ (None, 21)             │         2,121 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,221 (16.49 KB)

 Trainable params: 4,221 (16.49 KB)

 Non-trainable params: 0 (0.00 B)


Training the CBOW model...
Training complete.

Learned Word Embeddings (First 5 Dimensions shown):
Word: 'the       ' Vector: [-0.64037615 -0.51398855 -0.25916603  0.4341748   0.21438281]
Word: 'dog       ' Vector: [ 0.18356927 -0.28472608  0.03423562  0.18984587 -0.24708267]
Word: 'is        ' Vector: [ 0.25326893  0.11211544 -0.05640516  0.28516176  0.18400387]
Word: 'quick     ' Vector: [-0.18321015  0.11445179  0.5525027  -0.41065064 -0.43968046]
Word: 'brown     ' Vector: [-0.23953767 -0.003757   -0.10311186 -0.16485009 -0.6573329 ]
Word: 'fox       ' Vector: [ 0.02076282 -0.15911838  0.22037296 -0.14519896 -0.48222628]
Word: 'jumps     ' Vector: [ 0.21481562 -0.06030675  0.4145461  -0.548988   -0.42038766]
Word: 'over      ' Vector: [-0.49802524 -0.23029177  0.03903196  0.33053845 -0.38465258]
Word: 'lazy      ' Vector: [ 0.35861576 -0.08716509  0.33345026 -0.4409552  -0.6072849 ]
Word: 'a         ' Vector: [-0.53307855 -0.6367042  -0.12878607  0.500887    0.22777629]
